# Path Resolving (with array fan-out)

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Live Screen, Onsite Loop, Trees · **Difficulty/Frequency:** Uncommon (3/10)

> **Compare with** [`7. Deep_Key_Search_Nested_JSON`](../7.%20Deep_Key_Search_Nested_JSON/7.%20Deep_Key_Search_Nested_JSON.ipynb): that one searches for a key at **any** depth. This one follows an **exact** path — but fans out across arrays.

## Concepts

**What this problem is really testing:**
- Handling a structure with **two different kinds of internal node** (dict and list) that behave differently
- **Fan-out recursion**: one call producing *many* results, not one
- Precision about *when* the path advances — the single rule the whole problem turns on

**First-principles primer — what is each piece?**

- **Path.** A list of keys to follow in order, like `["a", "b", "c"]`. Same idea as MongoDB's dotted field notation `a.b.c`.
- **Fan-out.** In a plain nested dict, a path leads to at most **one** value. Add arrays and a single path can land on **many** values, because an array means "try all of these". So the return type has to be a *list*, even when only one thing matches.
- **The index that does — and does not — advance.** This is the entire problem:

| Current node | What the segment means | Path index |
|---|---|---|
| **dict** | look up this key | **advances** (`idx + 1`) |
| **list** | apply the *same* segment to every element | **stays** (`idx`) |
| primitive | you cannot index into a string or number | no match, return `[]` |

An array is **not a level of the path** — it is a *container* the path passes straight through. Grasping that one sentence is the difference between a solution that works and one that quietly loses results.

- **`extend` vs `append`.** Each recursive call returns a *list of matches*. Combining them needs `extend` (splice the items in) not `append` (nest the whole list). This is exactly what the last worked example in the prompt is testing.

**The subtlety in the final example.** The expected answer is `["foo", "bar", "blah", ["baz"]]` — and the prompt explicitly notes *"we're not returning `"baz"` by itself here."*

Why: the last `c` key holds the value `["baz"]`. Once the path is fully consumed, whatever you have landed on **is the answer**, verbatim. It happens to be a list, but that is the *matched value*, not a container to fan out through. Fan-out applies only to arrays encountered **while the path still has segments left**.

**Simple worked example.** `{"a": {"b": [{"c": "foo"}, {"c": "baz"}]}}` with path `["a", "b", "c"]`:

```
resolve(root, idx=0)              dict, key "a" present  -> descend, idx=1
  resolve({"b": [...]}, idx=1)    dict, key "b" present  -> descend, idx=2
    resolve([...], idx=2)         LIST -> fan out, idx STAYS 2
      resolve({"c":"foo"}, 2)     dict, key "c"          -> descend, idx=3
        resolve("foo", 3)         idx == len(path)       -> ["foo"]
      resolve({"c":"baz"}, 2)     dict, key "c"          -> descend, idx=3
        resolve("baz", 3)         idx == len(path)       -> ["baz"]
    -> ["foo", "baz"]             (extend, not append)
```

## Problem Statement

Given a JSON value (dict, list, or primitive) and a path (a list of string keys), return a **list of every value** reachable by that path.

**The rules**

- A **dict** consumes one path segment.
- A **list** consumes **none** — the current segment applies to every element, and results concatenate.
- A **primitive** with path left over matches nothing.
- When the path is exhausted, the current node **is** a match, whatever its type.

**Examples**

```python
resolve_path("foo", ["a"])                                  # -> []
resolve_path({"a": "foo"}, ["a"])                           # -> ["foo"]
resolve_path({"a": {"b": "foo"}}, ["a"])                    # -> [{"b": "foo"}]

resolve_path({"a": {"b": [{"c": "foo"}, {"c": "baz"}]}}, ["a", "b", "c"])
# -> ["foo", "baz"]

resolve_path(
    {"a": [{"b": [{"c": "foo"}, {"c": "bar"}]},
           {"b": [{"c": "blah"}, {"c": ["baz"]}]}]},
    ["a", "b", "c"])
# -> ["foo", "bar", "blah", ["baz"]]      <- NOT "baz" on its own
```

### Approach 1 — Naive (walk down a level at a time, flattening as you go)

**Idea:** hold a *frontier* — the list of nodes currently reachable. For each path segment, build the next frontier by looking that key up in every dict in the frontier, flattening any lists you meet on the way.

This is a breadth-first reading of the same rules and it *is* correct. The instructive part is what it costs: because lists can nest arbitrarily (`[[{...}]]`), each step needs its own flattening loop, and the "am I flattening a container or have I matched a value?" distinction has to be re-derived at every level rather than falling out of the structure.

**Time complexity:** O(V) over the explored portion of the tree.

**Space complexity:** **O(width)** — the whole frontier is materialised at every level, which can be much larger than the recursion depth.

In [ ]:
from typing import Any, Dict, Iterator, List, Optional


def _flatten_containers(nodes: List[Any]) -> List[Any]:
    """Expand any lists in `nodes` into their elements, recursively."""
    out: List[Any] = []
    for n in nodes:
        if isinstance(n, list):
            out.extend(_flatten_containers(n))      # a list is a container to pass THROUGH
        else:
            out.append(n)
    return out


def resolve_path_iterative(data: Any, path: List[str]) -> List[Any]:
    frontier = [data]
    for segment in path:
        frontier = _flatten_containers(frontier)    # arrays do not consume a segment
        nxt: List[Any] = []
        for node in frontier:
            if isinstance(node, dict) and segment in node:
                nxt.append(node[segment])           # a dict DOES consume the segment
        frontier = nxt
    # The path is exhausted, so whatever we landed on is a match VERBATIM - no final
    # flatten here, or a matched list value like ["baz"] would be torn apart.
    return frontier

### Approach 2 — Optimal (recursive fan-out)

**Idea:** one function, `resolve(node, idx)`, returning *all* values reachable from `node` by the remaining path `path[idx:]`. Four cases, each one line of reasoning:

1. **`idx == len(path)`** — the path is used up, so this node is a match. Return `[node]`, **not** `node`, because every branch must return a list.
2. **dict** — if `path[idx]` is a key, descend into it with `idx + 1`. If not, return `[]`.
3. **list** — recurse into every element with the **same** `idx`, and `extend` the results.
4. **primitive with path left** — return `[]`; you cannot index into a string.

The recursion mirrors the data's own shape, so the fan-out rule is expressed exactly once rather than re-derived at each level. It is also **lazy in the useful sense**: it only ever visits branches the path actually reaches, unlike the frontier version which materialises every level in full.

**Time complexity:** O(V) over the *explored* portion of the tree — untouched branches cost nothing.

**Space complexity:** O(d + r) — recursion depth `d` (path length plus array nesting) plus the result size `r`.

In [ ]:
def resolve_path(data: Any, path: List[str]) -> List[Any]:
    def resolve(node: Any, idx: int) -> List[Any]:
        if idx == len(path):
            return [node]                      # path exhausted: THIS node is the answer, as-is

        if isinstance(node, dict):
            key = path[idx]
            if key in node:
                return resolve(node[key], idx + 1)   # a dict CONSUMES the segment
            return []                                # key absent: nothing matches down here

        if isinstance(node, list):
            results: List[Any] = []
            for item in node:
                results.extend(resolve(item, idx))   # SAME idx - a list consumes nothing
            return results                           # extend, not append: keep the list flat

        return []                                    # a primitive with path left over

    return resolve(data, 0)

### Approach 3 — Iterative (an explicit stack, no recursion limit)

**Idea:** the same traversal with a stack of `(node, idx)` pairs, so deeply nested input cannot blow Python's ~1000-frame recursion limit.

**The one detail that matters:** results must come out in the same **document order** as the recursive version. A LIFO stack pops in reverse of the order you push, so list elements go on **reversed** — then they pop left-to-right and matches accumulate in document order with no final fix-up. Drop the `reversed()` and the code still "works": it just returns a scrambled order, which is exactly the kind of bug that survives casual testing and fails on real data.

**Time complexity:** O(V) over the explored portion.

**Space complexity:** O(V) worst case for the stack — slightly worse than the recursion's O(d), which is the price of not being able to overflow.

In [ ]:
def resolve_path_stack(data: Any, path: List[str]) -> List[Any]:
    stack: List[Any] = [(data, 0)]
    out: List[Any] = []
    while stack:
        node, idx = stack.pop()
        if idx == len(path):
            out.append(node)
            continue
        if isinstance(node, dict):
            key = path[idx]
            if key in node:
                stack.append((node[key], idx + 1))
        elif isinstance(node, list):
            # reversed(): LIFO would otherwise visit elements right-to-left
            for item in reversed(node):
                stack.append((item, idx))            # same idx
    return out

### Follow-up — wildcards

**Idea:** a `*` segment means "any key" in a dict, or "any element" in a list.

The dict case is the only real change: instead of one key lookup, recurse into **every** value with `idx + 1`. Note that `*` *does* consume a segment for a dict (it stands in for one key) but — like any segment — consumes nothing for a list.

This is the same semantics as JSONPath's `$.a.*.c` and MongoDB's `$[]` positional-all operator, and it composes with the fan-out rule for free: the recursion already knows how to produce many results from one call.

**Time complexity:** O(V) — a wildcard can force exploration of the whole subtree.

**Space complexity:** O(d + r).

In [ ]:
WILDCARD = "*"


def resolve_path_wild(data: Any, path: List[str]) -> List[Any]:
    def resolve(node: Any, idx: int) -> List[Any]:
        if idx == len(path):
            return [node]

        if isinstance(node, dict):
            seg = path[idx]
            if seg == WILDCARD:
                results: List[Any] = []
                for v in node.values():            # "*" matches EVERY key...
                    results.extend(resolve(v, idx + 1))   # ...and still consumes a segment
                return results
            return resolve(node[seg], idx + 1) if seg in node else []

        if isinstance(node, list):
            results = []
            for item in node:
                results.extend(resolve(item, idx))  # lists consume nothing, wildcard or not
            return results

        return []

    return resolve(data, 0)

## Verification

Every example from the problem statement, then the cases that separate a correct implementation from a plausible one: the `["baz"]` non-flattening rule, empty containers, falsy values, and deep nesting.

In [ ]:
import random

IMPLS = [resolve_path, resolve_path_stack, resolve_path_iterative]

# --- Every example from the problem statement ---
for fn in IMPLS:
    assert fn("foo", ["a"]) == [], fn.__name__                        # a primitive, path left over
    assert fn({"a": "foo"}, ["a"]) == ["foo"], fn.__name__
    assert fn({"a": {"b": "foo"}}, ["a"]) == [{"b": "foo"}], fn.__name__   # an OBJECT can be a match

    doc = {"a": {"b": [{"c": "foo", "d": {}, "e": "bar"}]}}
    assert fn(doc, ["a", "b", "c"]) == ["foo"], fn.__name__

    doc2 = {"a": {"b": [{"c": "foo", "d": {}, "e": "bar"}, {"c": "baz"}]}}
    assert fn(doc2, ["a", "b", "c"]) == ["foo", "baz"], fn.__name__

    # THE example: nested arrays fan out, but a matched list value stays intact
    doc3 = {"a": [{"b": [{"c": "foo"}, {"c": "bar"}]},
                  {"b": [{"c": "blah"}, {"c": ["baz"]}]}]}
    assert fn(doc3, ["a", "b", "c"]) == ["foo", "bar", "blah", ["baz"]], (
        f"{fn.__name__}: a matched list value must NOT be flattened"
    )

# --- The non-flattening rule, isolated ---
for fn in IMPLS:
    assert fn({"a": ["x", "y"]}, ["a"]) == [["x", "y"]], (
        f"{fn.__name__}: path exhausted at a list -> the list IS the match"
    )
    assert fn({"a": [{"b": 1}, {"b": 2}]}, ["a", "b"]) == [1, 2], (
        f"{fn.__name__}: a list WITH path remaining -> fan out"
    )

# --- Missing keys, empty containers, primitives ---
for fn in IMPLS:
    assert fn({"a": 1}, ["z"]) == [], fn.__name__                     # missing key
    assert fn({}, ["a"]) == [], fn.__name__                           # empty dict
    assert fn([], ["a"]) == [], fn.__name__                           # empty list
    assert fn({"a": {}}, ["a", "b"]) == [], fn.__name__               # empty dict mid-path
    assert fn({"a": []}, ["a", "b"]) == [], fn.__name__               # empty list mid-path
    assert fn(42, ["a"]) == [], fn.__name__                           # a number
    assert fn(None, ["a"]) == [], fn.__name__                         # null
    assert fn({"a": "str"}, ["a", "b"]) == [], fn.__name__            # can't index into a string
    assert fn({"a": 1}, []) == [{"a": 1}], f"{fn.__name__}: an empty path matches the root"
    assert fn("foo", []) == ["foo"], fn.__name__

# --- Falsy values are real matches, not absences ---
for fn in IMPLS:
    falsy = {"a": {"zero": 0, "empty": "", "false": False, "null": None, "arr": [], "obj": {}}}
    assert fn(falsy, ["a", "zero"]) == [0], fn.__name__
    assert fn(falsy, ["a", "false"]) == [False], fn.__name__
    assert fn(falsy, ["a", "null"]) == [None], fn.__name__
    assert fn(falsy, ["a", "empty"]) == [""], fn.__name__
    assert fn(falsy, ["a", "arr"]) == [[]], fn.__name__
    assert fn(falsy, ["a", "obj"]) == [{}], fn.__name__

# --- Arrays nested directly inside arrays still consume no segment ---
for fn in IMPLS:
    assert fn({"a": [[{"b": 1}], [{"b": 2}, {"b": 3}]]}, ["a", "b"]) == [1, 2, 3], fn.__name__
    assert fn(["val1", "val2", {"a": "foo"}], ["a"]) == ["foo"], (
        f"{fn.__name__}: a top-level array fans out too"
    )

# --- Results come out in document order ---
ordered = {"a": [{"b": "1"}, {"b": "2"}, {"b": "3"}, {"b": "4"}]}
for fn in IMPLS:
    assert fn(ordered, ["a", "b"]) == ["1", "2", "3", "4"], f"{fn.__name__}: document order"

# --- The returned values are the LIVE sub-objects, not copies ---
shared = {"a": {"b": {"deep": 1}}}
got = resolve_path(shared, ["a", "b"])
assert got[0] is shared["a"]["b"], "matches must be references, not copies"

# --- Deep nesting: the stack version survives where recursion would not ---
deep: Any = {"leaf": "bottom"}
for _ in range(3000):
    deep = {"down": deep}
deep_path = ["down"] * 3000 + ["leaf"]
assert resolve_path_stack(deep, deep_path) == ["bottom"], "no recursion limit"
try:
    resolve_path(deep, deep_path)
except RecursionError:
    pass                      # expected - exactly why the iterative version exists
else:
    pass                      # a raised limit is fine too; the point is only that one CANNOT fail

# --- Wildcards ---
w = {"a": {"x": {"c": 1}, "y": {"c": 2}, "z": {"d": 3}}}
assert resolve_path_wild(w, ["a", "*", "c"]) == [1, 2], "'*' matches any key, and consumes a segment"
assert resolve_path_wild(w, ["a", "*"]) == [{"c": 1}, {"c": 2}, {"d": 3}]
assert resolve_path_wild({"a": [{"b": 1}, {"b": 2}]}, ["a", "*"]) == [1, 2], (
    "inside a list, '*' still applies per element"
)
# With no wildcard present, it must agree with the plain resolver
for doc, path in [(doc2, ["a", "b", "c"]), (doc3, ["a", "b", "c"]), (w, ["a", "x", "c"])]:
    assert resolve_path_wild(doc, path) == resolve_path(doc, path)


# --- Randomised: all three implementations must agree ---
def random_json(rng, depth=0):
    if depth > 3 or rng.random() < 0.3:
        return rng.choice(["s", 1, True, None, 0, ""])
    if rng.random() < 0.5:
        return {rng.choice("abc"): random_json(rng, depth + 1) for _ in range(rng.randint(0, 3))}
    return [random_json(rng, depth + 1) for _ in range(rng.randint(0, 3))]


rng = random.Random(47)
for _ in range(600):
    doc = random_json(rng)
    path = [rng.choice("abc") for _ in range(rng.randint(0, 3))]
    expected = resolve_path(doc, path)
    assert resolve_path_stack(doc, path) == expected, (doc, path)
    assert resolve_path_iterative(doc, path) == expected, (doc, path)
    assert resolve_path_wild(doc, path) == expected, (doc, path)

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **Wildcards.** Implemented above. The neat part is how little changed: the recursion already produced many results from one call, so `*` simply widens the dict case from one key to all of them. Note the asymmetry worth saying aloud — `*` **consumes** a segment in a dict (it stands for one key) but consumes **nothing** in a list, because lists never consume segments at all.
- **References vs copies.** The solution returns the live sub-objects, asserted above. That is usually right — it is what `dict[key]` does, and copying a large matched subtree would be expensive and surprising. But say it explicitly, because the caller can now mutate your input through the result. If isolation matters, `copy.deepcopy` at the base case, and note the cost.
- **Numeric indices in the path (`["a", 0, "b"]`).** A genuine design fork. Adding an integer branch that indexes a list *and advances* `idx` conflicts with the fan-out rule — the same list would sometimes consume a segment and sometimes not, depending on the segment's type. That is defensible (JSONPath does it), but it makes the semantics type-dependent and much harder to explain. Flag the tension rather than implementing it silently.
- **Bounding the result size.** Worst case, every leaf matches, so `r = O(V)` — a path of all-wildcards over a wide tree returns the entire leaf set. If a caller could pass an adversarial path, cap the result and return a "truncated" flag; unbounded fan-out is a denial-of-service vector in a query API.
- **Recursion depth.** `resolve_path_stack` exists for this, and the test above shows it surviving 3000 levels where the recursive version raises. The detail that bites people is **ordering**: a LIFO stack reverses everything, so children must be pushed reversed to preserve document order. The code still "works" without it — it just returns results scrambled, which passes a `sorted()` comparison and fails a real one.
- **How this differs from a deep key search.** [`7. Deep_Key_Search_Nested_JSON`](../7.%20Deep_Key_Search_Nested_JSON/7.%20Deep_Key_Search_Nested_JSON.ipynb) asks "find this key **anywhere**" — it descends into every value regardless of key, and stops at the first hit. This one follows an **exact** route and never wanders, but returns *everything* that route reaches. Search versus resolve: one is a scan with an early exit, the other is a directed walk with fan-out.

## Empirical complexity check

**A result worth reporting honestly.** The first version of this benchmark grew the document *width* — thousands of sibling keys the path never visits — expecting the frontier approach to degrade. It did not: all three stayed flat.

The reason is that none of them ever touch an unreachable branch. The frontier only ever holds nodes the path actually led to, so siblings that fail the key lookup are never added to it in the first place. Document size is simply not a variable for any of these implementations — which is itself the useful finding: **cost scales with the size of the answer, not the size of the input.**

So the benchmark below grows what *does* matter: the **fan-out width**, i.e. how many array elements match. All three should be linear in the number of results.

| Growth when the match count doubles | What it means |
|---|---|
| ~2x | linear in the **output** size — the only thing any of these pay for |

**Where the three genuinely differ is space, not time:**

| | peak extra space | survives deep nesting? |
|---|---|---|
| frontier | O(width of a level) | yes — no recursion |
| recursive | O(depth) | **no** — `RecursionError` past ~1000 levels |
| explicit stack | O(nodes pending) | yes |

That is the real basis for choosing between them, and the assertions above test the recursion-limit difference directly.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

QUERIES = 200
PATH = ["target", "items", "value"]


def make_document(n):
    """n array elements that all match, so the RESULT size grows with n."""
    doc = {"target": {"items": [{"value": i} for i in range(n)]}}
    for i in range(50):
        doc[f"noise_{i}"] = {"items": [{"value": -1} for _ in range(5)]}   # never reached
    return (doc,)


def run_frontier(doc):
    for _ in range(QUERIES):
        resolve_path_iterative(doc, PATH)


def run_recursive(doc):
    for _ in range(QUERIES):
        resolve_path(doc, PATH)


def run_stack(doc):
    for _ in range(QUERIES):
        resolve_path_stack(doc, PATH)


benchmark(
    {"Approach 1 - level frontier": run_frontier,
     "Approach 2 - recursive fan-out": run_recursive,
     "Approach 3 - explicit stack": run_stack},
    make_document,
    sizes=[500, 1000, 2000, 4000],
    repeats=2,
)

## Patterns learned

- **Let the recursion mirror the data's shape.** Dicts, lists and primitives each get one branch, and the awkward rule ("lists do not consume a segment") is stated exactly once instead of re-derived at every level.
- **Decide precisely when your cursor advances.** One index, two behaviours: `idx + 1` for a dict, `idx` for a list. Nearly every bug in this problem is that line being wrong.
- **A function returning "all matches" must return a list from *every* branch.** No match is `[]`, one match is `[node]`. Returning a bare value from one branch and a list from another is the most common way this breaks.
- **`extend` combines, `append` nests.** When each recursive call yields a *collection* of results, `extend` is what keeps the answer flat.
- **A container is not a value — until the path runs out.** The same list is a thing to walk *through* mid-path and a thing to *return whole* at the end. That distinction is the entire point of the `["baz"]` example.
- **Falsy is not absent.** `0`, `""`, `False`, `None`, `[]` and `{}` are all legitimate matches. Branch on `key in node`, never on truthiness — the same trap as the sentinel in the deep-key-search problem.
- **When a stack replaces recursion, push children reversed.** LIFO flips the order; reversing flips it back. Silent bug otherwise.